In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os

DRIVE_FOLDER = '/content/drive/MyDrive/olist_project'
DATA_FOLDER = DRIVE_FOLDER
DB_PATH = os.path.join(DRIVE_FOLDER, 'olist.db')

os.makedirs(DRIVE_FOLDER, exist_ok=True)

In [4]:
from sqlalchemy import create_engine
import pandas as pd
engine = create_engine(f'sqlite:///{DB_PATH}')

In [5]:
for file in os.listdir(DATA_FOLDER):
    print(file)

olist_geolocation_dataset.csv
olist_customers_dataset.csv
olist_order_payments_dataset.csv
olist_orders_dataset.csv
olist_order_items_dataset.csv
product_category_name_translation.csv
olist_order_reviews_dataset.csv
olist_products_dataset.csv
olist_sellers_dataset.csv
olist.db


In [8]:
#converting the csv files to tables
csv_to_table = {
    'olist_customers_dataset.csv': 'customers',
    'olist_orders_dataset.csv': 'orders',
    'olist_order_items_dataset.csv': 'order_items',
    'olist_order_payments_dataset.csv': 'payments',
    'olist_order_reviews_dataset.csv': 'reviews',
    'olist_products_dataset.csv': 'products',
    'olist_sellers_dataset.csv': 'sellers',
    'olist_geolocation_dataset.csv': 'geolocation',
    'product_category_name_translation.csv': 'category_translation',
}

In [10]:
for csv_file, table_name in csv_to_table.items():
    csv_path = os.path.join(DATA_FOLDER, csv_file)
    if os.path.exists(csv_path):
        df = pd.read_csv(csv_path)
        df.to_sql(table_name, engine, if_exists='replace', index=False)
        print(f"Loaded {table_name}: {len(df)} rows")
    else:
        print(f"WARNING: {csv_file} not found at {csv_path}")

Loaded customers: 99441 rows
Loaded orders: 99441 rows
Loaded order_items: 112650 rows
Loaded payments: 103886 rows
Loaded reviews: 99224 rows
Loaded products: 32951 rows
Loaded sellers: 3095 rows
Loaded geolocation: 1000163 rows
Loaded category_translation: 71 rows


Exploratory Data Analysis

In [24]:
tables = pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table'", conn)
tables

,name
0,customers
1,orders
2,order_items
3,payments
4,reviews
5,products
6,sellers
7,geolocation
8,category_translation


In [21]:
tables = ['customers', 'orders', 'order_items', 'payments', 'reviews', 'products', 'sellers', 'geolocation', 'category_translation']
for t in tables:
    count = pd.read_sql_query(f"SELECT COUNT(*) as cnt FROM {t}", engine)
    print(f"{t}: {count['cnt'][0]} rows")

customers: 99441 rows
orders: 99441 rows
order_items: 112650 rows
payments: 103886 rows
reviews: 99224 rows
products: 32951 rows
sellers: 3095 rows
geolocation: 1000163 rows
category_translation: 71 rows


2. Date range of the dataset

In [25]:
pd.read_sql_query("""SELECT MIN(order_purchase_timestamp) AS earliest_order,
                      MAX(order_purchase_timestamp) AS latest_order
                      FROM orders""", engine)

,earliest_order,latest_order
0,2016-09-04 21:15:19,2018-10-17 17:30:18


3. Order status

In [26]:
pd.read_sql_query("""
    SELECT order_status, COUNT(*) AS num_orders
    FROM orders
    GROUP BY order_status
    ORDER BY num_orders DESC
""", engine)

,order_status,num_orders
0,delivered,96478
1,shipped,1107
2,canceled,625
3,unavailable,609
4,invoiced,314
5,processing,301
6,created,5
7,approved,2


4. Review score distribution

In [28]:
pd.read_sql_query("""
    SELECT review_score, COUNT(*) AS num_reviews
    FROM reviews
    GROUP BY review_score
    ORDER BY review_score
""", engine)

,review_score,num_reviews
0,1,11424
1,2,3151
2,3,8179
3,4,19142
4,5,57328


5. Basic payment check

In [29]:
pd.read_sql_query("""
    SELECT
        MIN(payment_value) AS min_payment,
        MAX(payment_value) AS max_payment,
        ROUND(AVG(payment_value), 2) AS avg_payment
    FROM payments
""", engine)

,min_payment,max_payment,avg_payment
0,0.0,13664.08,154.1


Business Questions
1. Do repeat customers spend more than customers who purchase only once?

In [1]:
q1 = """
SELECT c.customer_unique_id,
       COUNT(o.order_id) AS num_orders,
       SUM(p.payment_value) AS total_spend
FROM orders o
JOIN customers c ON o.customer_id = c.customer_id
JOIN payments p ON o.order_id = p.order_id
GROUP BY c.customer_unique_id
"""
df_q1 = pd.read_sql_query(q1, engine)

# do the "repeat vs one-time" split in pandas instead of SQL — simpler to explain
df_q1['customer_type'] = df_q1['num_orders'].apply(lambda x: 'Repeat' if x > 1 else 'One-time')
df_q1.groupby('customer_type')['total_spend'].mean()

NameError: name 'pd' is not defined

In [31]:
# ============================================
# Q2: Delivery time vs review score
# ============================================
q2 = """
SELECT
    r.review_score,
    COUNT(*) AS num_orders,
    ROUND(AVG(julianday(o.order_delivered_customer_date) - julianday(o.order_purchase_timestamp)), 1) AS avg_delivery_days
FROM orders o
JOIN reviews r ON o.order_id = r.order_id
WHERE o.order_delivered_customer_date IS NOT NULL
GROUP BY r.review_score
ORDER BY r.review_score;
"""
result_q2 = pd.read_sql_query(q2, engine)
print(result_q2)

   review_score  num_orders  avg_delivery_days
0             1        9409               21.3
1             2        2941               16.7
2             3        7962               14.3
3             4       18987               12.3
4             5       57060               10.7


In [32]:
# ============================================
# Q3: Revenue by category over time
# ============================================
q3 = """
SELECT
    strftime('%Y-%m', o.order_purchase_timestamp) AS order_month,
    ct.product_category_name_english AS category,
    ROUND(SUM(oi.price), 2) AS revenue,
    COUNT(DISTINCT oi.order_id) AS num_orders
FROM order_items oi
JOIN orders o ON oi.order_id = o.order_id
JOIN products p ON oi.product_id = p.product_id
JOIN category_translation ct ON p.product_category_name = ct.product_category_name
GROUP BY order_month, category
ORDER BY order_month, revenue DESC;
"""
result_q3 = pd.read_sql_query(q3, engine)
print(result_q3.head(20))

   order_month               category  revenue  num_orders
0      2016-09          health_beauty   134.97           1
1      2016-09        furniture_decor    72.89           1
2      2016-09              telephony    59.50           1
3      2016-10        furniture_decor  5807.89          51
4      2016-10              perfumery  5688.70          30
5      2016-10          health_beauty  4552.51          44
6      2016-10                   toys  4465.09          25
7      2016-10         consoles_games  3882.26          10
8      2016-10          watches_gifts  3360.24           5
9      2016-10         sports_leisure  3333.64          19
10     2016-10                   auto  1833.25          11
11     2016-10       air_conditioning  1707.09           5
12     2016-10                   baby  1630.16          11
13     2016-10  computers_accessories  1399.32          18
14     2016-10           garden_tools  1359.88           5
15     2016-10            electronics  1306.99          